### Carga de Datos:

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
from preprocessing_utils import preprocess_text
from features.extraction import FeatureExtractor
from models.train_classic import train_logistic
from models.train_transformer import run_finetuning

# 1. Cargar Datos
train = pd.read_csv('../data/processed/train.csv')
test = pd.read_csv('../data/processed/test.csv')


print(f"Train: {len(train)} | Test: {len(test)}")


INFO:preprocessing_utils:Modelo spaCy cargado correctamente
INFO:preprocessing_utils:Descargando recurso NLTK: stopwords
c:\Users\DELL\Desktop\TT\proyecto-transformacion-texto-imagen\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 908 | Test: 114


### **Experimento 1.1:** Limpieza Básica + TF-IDF + Regresión Logística

In [ ]:
# Usamos P1 (Limpieza mínima)
X_train_p1 = [preprocess_text(t, 'P1') for t in train['text']]
X_test_p1 = [preprocess_text(t, 'P1') for t in test['text']]

extractor = FeatureExtractor()
# Vectorizamos con TF-IDF
X_train_tfidf, vectorizer = extractor.get_tfidf(X_train_p1)
X_test_tfidf = vectorizer.transform(X_test_p1)

# Entrenamos con Regresión Logística
train_logistic(X_train_tfidf, train['manual_classification'], 
               X_test_tfidf, test['manual_classification'], "Exp_1.1_TFIDF_Baseline")


Entrenando Regresión Logística para Exp_1.1_TFIDF_Baseline...
--- Resultados Exp_1.1_TFIDF_Baseline ---
              precision    recall  f1-score   support

           0       0.76      0.74      0.75        70
           1       0.61      0.64      0.62        44

    accuracy                           0.70       114
   macro avg       0.69      0.69      0.69       114
weighted avg       0.70      0.70      0.70       114



(LogisticRegression(class_weight='balanced', random_state=42, solver='liblinear'),
 array([1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,
        1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1,
        1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0,
        1, 0, 1, 0]))

### **Experimento 1.2:** Limpieza Básica + Stopwords + Lematización + TF-IDF + Regresión Logística

In [3]:
# Usamos P2 (Quitamos stopwords)
X_train_p2 = [preprocess_text(t, 'P2') for t in train['text']]
X_test_p2 = [preprocess_text(t, 'P2') for t in test['text']]

# Vectorizamos con TF-IDF
X_train_tfidf_sw, vec_sw = extractor.get_tfidf(X_train_p2)
X_test_tfidf_sw = vec_sw.transform(X_test_p2)

# Entrenamos con Regresión Logística
train_logistic(X_train_tfidf_sw, train['manual_classification'], 
               X_test_tfidf_sw, test['manual_classification'], "Exp_1.2_TFIDF_Stopwords")


Entrenando Regresión Logística para Exp_1.2_TFIDF_Stopwords...
--- Resultados Exp_1.2_TFIDF_Stopwords ---
              precision    recall  f1-score   support

           0       0.78      0.73      0.76        70
           1       0.61      0.68      0.65        44

    accuracy                           0.71       114
   macro avg       0.70      0.71      0.70       114
weighted avg       0.72      0.71      0.71       114



(LogisticRegression(class_weight='balanced', random_state=42, solver='liblinear'),
 array([1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
        1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1,
        1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0,
        1, 0, 1, 1]))

### **Experimento 2.2:** Limpieza Básica + RoBERTa-BNE (Frozen) + Regresión Logística

In [ ]:
# ==========================================
# EXP 2.2: Frozen RoBERTa + LR
# ==========================================
# Usamos P1 (Transformers prefieren texto con estructura, no P2)
print("Generando embeddings con RoBERTa (esto puede tardar)...")
X_train_emb = extractor.get_transformer_embeddings(X_train_p1, "PlanTL-GOB-ES/roberta-base-bne")
X_test_emb = extractor.get_transformer_embeddings(X_test_p1, "PlanTL-GOB-ES/roberta-base-bne")

train_logistic(X_train_emb, train['manual_classification'], 
               X_test_emb, test['manual_classification'], "Exp_2.2_RoBERTa_Frozen")

### **Experimento 3.2:** Limpieza Básica + RoBERTa-BNE(Fine-Tuning) + Softmax

In [ ]:
# ==========================================
# EXP 3.2: Fine-Tuning RoBERTa
# ==========================================
# Este es el pesado. Asegúrate de estar en GPU.
trainer = run_finetuning(X_train_p1, train['manual_classification'].values, 
                         X_test_p1, test['manual_classification'].values, 
                         "PlanTL-GOB-ES/roberta-base-bne")